# M3L2 E07 - Retriever (Resolution)


In [ ]:
import os, getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
DOCS = [
    "La politica de vacaciones es de 15 dias por ano.",
    "El seguro medico esta incluido desde el primer dia.",
    "El horario de trabajo es de 9 a 18 con almuerzo.",
    "El trabajo remoto esta permitido 3 dias por semana.",
    "Los bonos anuales se pagan en diciembre.",
]
vectorstore = FAISS.from_texts(DOCS, embeddings)
print("Vector store listo.")


In [ ]:
# similarity_search directo
docs_direct = vectorstore.similarity_search("vacaciones", k=2)
print("similarity_search:")
for doc in docs_direct:
    print(f"  - {doc.page_content}")


In [ ]:
# TODO 1
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"Tipo del retriever: {type(retriever).__name__}")


In [ ]:
# TODO 2
docs_retriever = retriever.invoke("vacaciones")
print("as_retriever + invoke:")
for doc in docs_retriever:
    print(f"  - {doc.page_content}")
print()
print("Mismo resultado. Si cambias FAISS por Chroma, solo cambia la linea del vectorstore.")


In [ ]:
# TODO 3
retriever_k1 = vectorstore.as_retriever(search_kwargs={"k": 1})
retriever_k3 = vectorstore.as_retriever(search_kwargs={"k": 3})
query = "politica de empresa"
print(f"k=1: {len(retriever_k1.invoke(query))} doc(s)")
print(f"k=3: {len(retriever_k3.invoke(query))} doc(s)")
print("Tradeoff: k mayor = mas contexto pero mas tokens")


In [ ]:
def run_checks():
    assert retriever is not None
    docs = retriever.invoke("vacaciones")
    assert isinstance(docs, list) and len(docs) > 0
    assert len(docs) <= 2
    docs_remoto = retriever.invoke("trabajo desde casa")
    assert any("remoto" in d.page_content.lower() for d in docs_remoto)
    print("M3L2 E07 Resolution checks passed")

run_checks()
